In [5]:
# ========================================================
#  COMPONENT 2 - Vector Store Manager (ChromaDB)
# ========================================================

from langchain_community.embeddings import HuggingFaceEmbeddings

class VectorStoreManager:
    """Manages ChromaDB: embedding, upserting, and retrieval of chunks."""

    def __init__(
        self,
        persist_directory: str = "/content/chroma_db",
        collection_name: str = "smart_contracts",
    ):
        self.persist_directory = persist_directory
        self.collection_name   = collection_name
        self.vectorstore: Optional[Chroma] = None

        # Local sentence-transformers model — no API key, no quota issues
        print("Loading embedding model (first run downloads ~90MB)...")
        self.embeddings = HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-MiniLM-L6-v2",
            model_kwargs={"device": "cpu"},
            encode_kwargs={"normalize_embeddings": True},
        )
        print("\n\u2705 Embedding model ready.")
        os.makedirs(persist_directory, exist_ok=True)

    def add_documents(self, chunks: List[Document]) -> int:
        """Embed and store chunks in ChromaDB with deduplication."""
        if self.vectorstore is None:
            self.vectorstore = Chroma(
                collection_name=self.collection_name,
                embedding_function=self.embeddings,
                persist_directory=self.persist_directory,
            )
        ids = [
            f"{c.metadata.get('source', 'doc')}__chunk{c.metadata.get('chunk_id', i)}"
            for i, c in enumerate(chunks)
        ]
        self.vectorstore.add_documents(documents=chunks, ids=ids)
        return len(chunks)

    def get_retriever(self, top_k: int = 5):
        """Return a LangChain retriever for semantic similarity search."""
        if self.vectorstore is None:
            raise RuntimeError("No documents ingested yet. Upload a contract first.")
        return self.vectorstore.as_retriever(
            search_type="similarity", search_kwargs={"k": top_k}
        )

    def clear(self):
        if self.vectorstore is not None:
            self.vectorstore.delete_collection()
            self.vectorstore = None

    @property
    def document_count(self) -> int:
        return self.vectorstore._collection.count() if self.vectorstore else 0


print("\n\u2705 VectorStoreManager defined.")


✅ VectorStoreManager defined.
